# 07 · AI quality: MLflow evaluation of the deployed app, tracked over time (executed)

10 store-manager questions with 32 expected facts (`scripts/eval_dataset.json`), scored by MLflow judges
(Correctness, RelevanceToQuery, Safety) against the **deployed app** end to end. The daily ops job re-runs
this as the `kaapi-eval-runner` service principal and appends a row to `eval_history`. This notebook runs it
once more now, then shows the history and per-question results.

In [1]:
import warnings; warnings.filterwarnings('ignore')
from proof_helpers import *
print('catalog.schema:', FQ)

catalog.schema: fevm_cme_conde_catalog.kaapi_bricks


In [2]:
pin_mlflow_credentials()
import evaluate_app
run_id, metrics, results = evaluate_app.main(source="notebook")

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


2026/10/23 23:55:16 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.


Evaluating:   0%|          | 0/10 [Elapsed: 00:00, Remaining: ?]

Evaluating:  10%|█         | 1/10 [Elapsed: 02:53, Remaining: 26:04]

Evaluating:  20%|██        | 2/10 [Elapsed: 03:19, Remaining: 13:18]

...

Evaluating:  90%|█████████ | 9/10 [Elapsed: 03:36, Remaining: 00:24]

Evaluating: 100%|██████████| 10/10 [Elapsed: 03:38, Remaining: 00:00]

Evaluating: 100%|██████████| 10/10 [Elapsed: 03:38, Remaining: 00:00] [predict_fn: 91%, scorers: 9%]

Evaluating: 100%|██████████| 10/10 [Elapsed: 03:38, Remaining: 00:00] [predict_fn: 91%, scorers: 9%]

run_id cad908b6a24e4613906ae5f2c22a0bf9
  correctness/mean: 1.000
  relevance_to_query/mean: 1.000
  safety/mean: 1.000


In [3]:
h = df(f"SELECT evaluated_at, source, n_questions, correctness, relevance, safety, mlflow_run_id FROM {FQ}.eval_history ORDER BY evaluated_at")
print(h.to_string(index=False))

            evaluated_at    source n_questions correctness relevance safety                    mlflow_run_id
2026-10-23T18:14:04.000Z scheduled          10         0.9       1.0    1.0 ebb37806d793445e97afc05e411eb18c
2026-10-23T18:23:51.000Z  notebook          10         1.0       1.0    1.0 188687d80bbf47799752b735447855a7
2026-10-23T18:29:40.000Z  notebook          10         1.0       1.0    1.0 cad908b6a24e4613906ae5f2c22a0bf9


In [4]:
r = results.result_df
score_cols = [c for c in r.columns if c.endswith("/value")]
out = pd.DataFrame({"question": r["request"].map(lambda q: (q.get("query") if isinstance(q, dict) else str(q))[:62])})
for c in score_cols:
    out[c.split("/")[0]] = r[c]
out["answer_start"] = r["response"].map(lambda a: " ".join(str(a).split())[:70])
print(out.to_string(index=False))

                                                     question  expected_facts safety correctness relevance_to_query                                                           answer_start
                       How do I make a Classic Filter Coffee?             NaN    yes         yes                yes ## Classic Filter Coffee (PRD-001) — Rs.60 This is Kaapi Bricks' best-
              What is the correct decoction ratio and timing?             NaN    yes         yes                yes ## Decoction Ratio & Timing (Kaapi Bricks House Blend) **Coffee-to-wat
What should a barista do if a customer reports a nut allergy?             NaN    yes         yes                yes ## Barista Allergen Protocol — Nut Allergy Per Kaapi Bricks' Food Safe
                 How long does Coorg Arabica take to reorder?             NaN    yes         yes                yes ## Coorg Arabica Reorder Lead Time — Koramangala (STR-001) **Lead time
                   At what temperature should milk be stored?    